In [1]:
print("okay")

okay


In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'/home/jayant/Music/AI Projects/Medical-Chatbot-with-LLM'

In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter

/home/jayant/anaconda3/envs/medibot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def load_pdf_files(data) :
    loader = DirectoryLoader(
        data, 
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [8]:
extracted_data = load_pdf_files("data")

In [9]:
len(extracted_data)

637

In [10]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [11]:
minmal_docs = filter_to_minimal_docs(extracted_data)

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#split thee documents into smaller chunks  
def text_split(minmal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20,
    )
    text_chunk =text_splitter.split_documents(minmal_docs)
    return text_chunk

In [18]:
text_chunk = text_split(minmal_docs)
print(f"no of chunks is : {len(text_chunk)}")

no of chunks is : 5860


In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

/tmp/ipykernel_11072/2533971096.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3675.77it/s]


In [24]:
vector = embedding.embed_query("hello")
vector

[-0.06277173012495041,
 0.0549587644636631,
 0.052164819091558456,
 0.08578997850418091,
 -0.0827488899230957,
 -0.07457297295331955,
 0.06855470687150955,
 0.018396418541669846,
 -0.08201132714748383,
 -0.037384867668151855,
 0.012124900706112385,
 0.0035183196887373924,
 -0.004134280141443014,
 -0.04378441348671913,
 0.021807292476296425,
 -0.005102707073092461,
 0.01954660378396511,
 -0.042348768562078476,
 -0.11035963147878647,
 0.0054245381616055965,
 -0.05573476850986481,
 0.02805243246257305,
 -0.023158719763159752,
 0.028481315821409225,
 -0.053709641098976135,
 -0.052601587027311325,
 0.03393927589058876,
 0.04538863152265549,
 0.02371843531727791,
 -0.07312081754207611,
 0.054777730256319046,
 0.01704728975892067,
 0.08136032521724701,
 -0.002862711902707815,
 0.011958064511418343,
 0.07355859130620956,
 -0.09423746913671494,
 -0.08136206120252609,
 0.04001542925834656,
 0.0006921681924723089,
 -0.013393261469900608,
 -0.05453808978199959,
 0.005151398945599794,
 -0.026139816

In [26]:
print("Vector Length :" ,len(vector))

Vector Length : 384


In [30]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [32]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPEN_AI_API_KEY = os.getenv("OPEN_AI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPEN_AI_API_KEY"] = OPEN_AI_API_KEY

In [33]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [35]:
pc

In [36]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [38]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents = text_chunk,
    embedding = embedding,
    index_name = index_name
    
)

In [39]:
dummy_data = Document(
    page_content= "This is a test document",
    metadata = {"source" : "Learning curve"}
)

In [41]:
docsearch.add_documents(documents=[dummy_data])

['26758516-3593-413e-928a-bcb7376eadcc']

In [42]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs = {"k" : 3} )

In [45]:
retreived_docs = retriever.invoke("What is acne")
retreived_docs

[Document(id='0cba8a40-236a-4d0a-94d2-db22b0e7e85b', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='2dd5de63-b060-420c-9114-be0729a9efce', metadata={'source': 'data/Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='9ac78cf4-acbc-43ca-b6e1-cc6cac44e670', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general

In [50]:

from langchain_openai import ChatOpenAI
chatmodel = ChatOpenAI(model="gpt-4o", api_key=OPEN_AI_API_KEY)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough